# ConvNeXt-Tiny: first canonical training acceptance

Stage 1B2: one frozen two-epoch experiment, using shared training/checkpoint modules.
**Real CUB training metrics, updates and checkpoints remain PENDING until this cloud run returns evidence.**
Local fits used synthetic CPU fixtures. This notebook contains no recorded training results.
Open in an online Colab GPU runtime first (Runtime > Change runtime type > GPU).
Run cells in order; inspect settings before the training cell. No official test pixels or tuning.


## 1. Inspect runtime and pin published source

Keep the working CUDA Torch/Torchvision stack. Do not install the CPU requirements here.
Set `PUBLISHED_COMMIT` to the full published PR SHA containing this notebook; the empty value intentionally fails.
Private access uses a hidden read-only token prompt, never a cell/URL/argument.
A SHA-specific checkout preserves previous source and global HF/Torch caches. If project modules were
already imported, restart the kernel before switching code; never reload Torch/CUDA packages.


In [ ]:
import importlib.metadata
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

print({"python": sys.version.split()[0], "platform": sys.platform})
import torch
try:
    import torchvision
except Exception as error:
    raise RuntimeError("Repair the installed CUDA Torch/Torchvision pair deliberately, restart, then rerun.") from error
print({"torch": torch.__version__, "torchvision": torchvision.__version__,
       "cuda_build": torch.version.cuda, "cuda_available": torch.cuda.is_available()})
if not torch.cuda.is_available():
    raise RuntimeError("GPU training is PENDING: select and connect a GPU runtime before continuing.")
print({"device_name": torch.cuda.get_device_name(0),
       "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory})


In [ ]:
def checkout_repository(root, url, commit, private=False):
    import getpass
    import os
    from pathlib import Path
    import re
    import subprocess
    import sys
    import tempfile
    from urllib.parse import urlsplit

    if not isinstance(commit, str) or re.fullmatch(r"[0-9a-fA-F]{40}", commit) is None:
        raise ValueError("Set the full 40-character published commit SHA before repository setup.")
    parsed = urlsplit(url)
    if parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError("Repository URL must be credential-free HTTPS without query or fragment.")
    root = Path(root).resolve()
    if root.exists():
        if not (root / ".git").exists():
            raise ValueError("Existing source must include .git provenance; use a fresh path or a clean pinned checkout.")
    else:
        env = dict(os.environ)
        env["GIT_TERMINAL_PROMPT"] = "0"
        token = None
        try:
            with tempfile.TemporaryDirectory(prefix="featherlens-askpass-") as temporary:
                if private:
                    if os.name != "posix":
                        raise RuntimeError("Private clone helper targets Colab/Kaggle Linux; attach a clean pinned Git checkout on other platforms.")
                    token = getpass.getpass("Read-only GitHub token (hidden): ")
                    if not token:
                        raise ValueError("A nonempty read-only token is required for this private clone.")
                    helper = Path(temporary) / "askpass.py"
                    helper.write_text("#!" + sys.executable + "\nimport os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['FEATHERLENS_GIT_TOKEN'])\n", encoding="utf-8")
                    helper.chmod(0o700)
                    env["GIT_ASKPASS"] = str(helper)
                    env["FEATHERLENS_GIT_TOKEN"] = token
                # No credentials in command arguments; suppress Git's remote error text.
                subprocess.run(["git", "-c", "credential.helper=", "clone", "--no-checkout", url, str(root)],
                               env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
                subprocess.run(["git", "-C", str(root), "checkout", "--detach", commit],
                               env=env, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        except subprocess.CalledProcessError:
            raise RuntimeError("Repository setup failed. Check access and the published SHA; remove only the incomplete clone before retrying.") from None
        finally:
            env.pop("FEATHERLENS_GIT_TOKEN", None)
            env.pop("GIT_ASKPASS", None)
            token = None
    actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
    if actual.lower() != commit.lower():
        raise ValueError("Existing checkout differs from the published commit; attach/check out the intended commit explicitly.")
    dirty = subprocess.check_output(["git", "-C", str(root), "status", "--porcelain", "--untracked-files=no"], text=True)
    if dirty.strip():
        raise ValueError("Use a clean published checkout; tracked files have local modifications.")
    return actual

CLOUD = "colab"  # Or "kaggle" with Internet enabled.
if CLOUD not in {"colab", "kaggle"}:
    raise ValueError("CLOUD must be colab or kaggle")
WORK_DIR = Path("/content" if CLOUD == "colab" else "/kaggle/working")
PUBLISHED_COMMIT = ""  # REQUIRED: full published Stage 1B2 PR commit SHA.
REPO_DIR = WORK_DIR / ("featherlens-" + PUBLISHED_COMMIT[:12])
REPO_URL = "https://github.com/AnonyBOSS/featherlens-pixels-to-language.git"
PRIVATE_REPOSITORY = True
SOURCE_COMMIT = checkout_repository(REPO_DIR, REPO_URL, PUBLISHED_COMMIT, private=PRIVATE_REPOSITORY)
if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError("Project modules already imported: restart the kernel and rerun with this published checkout.")
os.chdir(REPO_DIR)
if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))
print({"source_commit": SOURCE_COMMIT, "repository": str(REPO_DIR)})


## 2. Missing dependencies, canonical metadata and Drive

Review missing packages; enable installation only when needed. Existing versions are preserved and printed.
`mlflow-skinny==3.17.2` is the tested tracking dependency. Inspect every captured `pip check` conflict. Kaggle alone may continue with a visible warning
for seven reviewed unrelated image-package relationships; unknown conflicts and required-module import
failures still stop setup. Repair identified packages deliberately when needed. No whole-environment upgrades.
Upload the verified metadata ZIP and set `PREPARED_ZIP`, or place the two unchanged files in `PREPARED_DIR`.
The ZIP contains no pixels. Extraction reads only the two known files and refuses overwrites/ambiguity.
Canonical SHA/count/class validation happens before the shared trainer opens the pinned official-train source.
Colab mounts Drive for the persistent experiment; authorize the mount when prompted.


In [ ]:
INSTALL_MISSING_DEPS = False
required = {"datasets": ("datasets", "5.1.0"), "numpy": ("numpy", "2.5.4"),
            "PIL": ("Pillow", "13.0.0"), "fsspec": ("fsspec", "2026.7.0"),
            "mlflow": ("mlflow-skinny", "3.17.2")}
missing = [f"{package}=={version}" for module, (package, version) in required.items()
           if importlib.util.find_spec(module) is None]
print({"missing_non_torch_dependencies": missing})
if missing and not INSTALL_MISSING_DEPS:
    raise RuntimeError("Review missing packages, set INSTALL_MISSING_DEPS=True, and rerun this cell.")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *missing], check=True)
check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
print(check.stdout)
if check.stderr:
    print(check.stderr)
if check.returncode or check.stderr:
    import re

    # Only these reviewed, unrelated Kaggle image relationships may warn.
    reviewed_pairs = {("bigframes", "google-cloud-bigquery-storage"),
                      ("google-colab", "jupyter-server"), ("google-colab", "pandas"),
                      ("google-colab", "requests"), ("dopamine-rl", "gym"),
                      ("gradio", "pillow"), ("moviepy", "decorator")}
    lines = [line.strip() for line in check.stdout.splitlines() if line.strip()]
    matches = [re.fullmatch(
        r"([\w.-]+) \S+ (?:requires|has requirement) ([\w.-]+)(?:[<>=!~\[].*?)?, (?:which is not installed|but you have .+)\.",
        line) for line in lines]
    reviewed_kaggle_only = (
        CLOUD == "kaggle" and check.returncode == 1 and not check.stderr and bool(matches)
        and all(match and tuple(re.sub(r"[-_.]+", "-", name.lower()) for name in match.groups())
                in reviewed_pairs for match in matches)
    )
    if not reviewed_kaggle_only:
        raise RuntimeError("pip check found conflicts or process errors above. Repair identified packages deliberately before continuing.")
    print("WARNING: Kaggle image has only the reviewed unrelated package conflicts above. "
          "Continuing without package repairs; required training imports must still succeed.")
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
for module in required:
    importlib.import_module(module)  # Installed-but-broken required packages remain fatal.
versions = {}
for module, (package, _) in required.items():
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = importlib.metadata.version(module)  # Existing full mlflow distribution.
print(versions)
from src.training.runtime import validate_preflight_preparation


In [ ]:
def extract_prepared_metadata(archive, destination):
    from pathlib import Path, PurePosixPath
    import zipfile

    destination = Path(destination)
    names = ("preparation.json", "classification_manifest.jsonl")
    with zipfile.ZipFile(archive) as handle:
        selected = {}
        for name in names:
            matches = [entry for entry in handle.infolist()
                       if not entry.is_dir() and PurePosixPath(entry.filename).name == name]
            if len(matches) != 1:
                raise ValueError("Handoff ZIP must contain exactly one copy of each known preparation file.")
            selected[name] = matches[0]
        if any((destination / name).exists() for name in names):
            raise FileExistsError("Prepared metadata already exists; use it directly or choose an empty directory.")
        # Read only known entries, never extract archive paths or arbitrary files.
        payloads = {name: handle.read(entry) for name, entry in selected.items()}
    destination.mkdir(parents=True, exist_ok=True)
    for name, payload in payloads.items():
        with (destination / name).open("xb") as output:
            output.write(payload)

if CLOUD == "colab":
    from google.colab import drive
    drive.mount("/content/drive")

PREPARED_DIR = WORK_DIR / "classification-v1"
PREPARED_ZIP = None  # Example: Path("/content/verified-metadata.zip")
if PREPARED_ZIP is not None:
    known_files = [PREPARED_DIR / name for name in ("preparation.json", "classification_manifest.jsonl")]
    if not all(path.is_file() for path in known_files):
        extract_prepared_metadata(PREPARED_ZIP, PREPARED_DIR)
prepared_identity = validate_preflight_preparation(PREPARED_DIR)
print(json.dumps({key: value for key, value in prepared_identity.items() if key != "class_names"}, indent=2))
print({"ordered_class_count": len(prepared_identity["class_names"])})


## 3. Acceptance config and stable run directory

The JSON sets ConvNeXt-Tiny frozen for two epochs, physical batch 8, accumulation 2,
AdamW LR 0.001, decay 0.0001, seed 42, explicit FP16, workers 0 and frozen BN statistics.
Keep the baseline full-image white 224 canvas, ImageNet normalization and train-only flips.
Batch 8 is a conservative starting choice from T4 preflight headroom; actual capacity/time are measured here.
These are acceptance settings, not tuned performance claims. If all steps succeed, expect 600 optimizer
updates (`ceil(ceil(4794/8)/2) * 2`); actual history/summary and skipped AMP updates are authoritative.

Keep `RUN_NAME` unchanged for this experiment. For recovery select `RESUME=True` explicitly and keep the
same published code, canonical preparation, config, tracking mode and entire persistent run directory.
Change the name for a changed configuration. One writer per run directory; no silent reset/overwrite.
Completed runs return saved results without duplicate training. Recovery is at completed epochs;
fresh-session GPU/cross-version restoration has not been established by local CPU checks.


In [ ]:
from src.training.trainer import TrainConfig
from src.training.experiment import run_training

config = TrainConfig.from_dict(json.loads((REPO_DIR / "configs/pretrained_acceptance.json").read_text(encoding="utf-8")))
RUN_NAME = "convnext-frozen-acceptance-v1"
RUN_ROOT = Path("/content/drive/MyDrive/featherlens/experiments" if CLOUD == "colab" else "/kaggle/working/experiments")
RUN_DIR = RUN_ROOT / RUN_NAME
RESUME = False  # Set True explicitly only to reuse this run's latest completed checkpoint.
DEVICE = "cuda"
TRACKING = "mlflow"
print(json.dumps(config.to_dict(), indent=2))
print({"run_directory": str(RUN_DIR), "resume": RESUME, "tracking": TRACKING})


## 4. Train or explicitly resume

This single shared call opens only canonical train/validation pixels from the pinned official-training
source, verifies the official weights for a fresh run, trains and reloads the best saved checkpoint.
It records real runtime, validation aggregates, successful/skipped updates and local MLflow identity.
Do not launch another writer. On interruption preserve the directory; rerun setup/config with explicit
resume. An interruption before the first completed checkpoint cannot resume; retain failure evidence
and choose a new run name. No ETA is promised. Downloads and global caches can be reused.


In [ ]:
torch.cuda.reset_peak_memory_stats()
result = run_training(PREPARED_DIR, RUN_DIR, config, device=DEVICE, resume=RESUME, tracking=TRACKING)
torch.cuda.synchronize()
print({"peak_allocated_bytes_this_call": torch.cuda.max_memory_allocated(),
       "peak_reserved_bytes_this_call": torch.cuda.max_memory_reserved(),
       "status": result["status"], "actual_optimizer_updates": result["optimizer_updates"]})


## 5. Results and checkpoint proof

Read actual files even after a handled failure. Fractions such as accuracy/F1 are in [0, 1].
Inspect failures and tracking status as well as completion. A forward proof must say `forward_verified`
and identify the best checkpoint SHA; terminal report repair may instead say `state_loaded_without_forward`.
That state does not establish a new inference check. Preserve both checkpoints, SHA sidecars and `mlruns`.


In [ ]:
summary = json.loads((RUN_DIR / "run_summary.json").read_text(encoding="utf-8"))
for key in ("status", "epochs_completed", "optimizer_updates", "skipped_updates", "best_epoch",
            "best_metrics", "latest_metrics", "elapsed_seconds", "resolved_precision", "runtime",
            "checkpoint_reload", "tracking", "failures"):
    print(json.dumps({key: summary.get(key)}, indent=2))
for name, reference in summary.get("checkpoints", {}).items():
    print({"checkpoint": name, "path": str(RUN_DIR / reference["path"]), "sha256": reference["sha256"]})
tracking_location = summary.get("tracking", {}).get("location")
if tracking_location:
    print({"tracking_path": str(RUN_DIR / tracking_location)})


## 6. Return evidence and retain the experiment

Download **`run_summary.json` and `evidence.zip`** and return both for review, even on failure if available.
The compact ZIP includes JSON/CSV/config/log files and `sha256.json`; it intentionally excludes
checkpoints, weights, pixels, caches and MLflow storage. Keep the entire Drive run directory separately,
including latest/best checkpoints, their integrity sidecars, RNG/optimizer state and local tracking.
Never treat the small evidence ZIP as a recovery backup. Verify retained files after reconnecting;
immediate writes do not prove fresh-session durability. MLflow stores local artifact locations;
relocating its FileStore across machines is not guaranteed.

**Kaggle online:** import this notebook, enable Internet and a GPU accelerator, set `CLOUD="kaggle"`
and point metadata to `/kaggle/input/<dataset>/...` (or extract into `/kaggle/working`). No Drive mount.
`RUN_DIR` is under `/kaggle/working/experiments`; use **Save Version** to persist notebook outputs,
then download the entire run directory and verify the saved version. Session storage alone is temporary.
Restore the complete run to the same path for explicit resume; offline packaging is not provided here.
Clear outputs before sharing a source notebook. Real canonical training acceptance remains pending
until returned cloud artifacts establish the actual outcome.


In [ ]:
return_files = [RUN_DIR / "run_summary.json", RUN_DIR / "evidence.zip"]
for path in return_files:
    print({"return_file": str(path), "exists": path.is_file()})
print({"preserve_entire_run_directory": str(RUN_DIR)})
DOWNLOAD_COLAB_EVIDENCE = False
if CLOUD == "colab" and DOWNLOAD_COLAB_EVIDENCE:
    from google.colab import files
    for path in return_files:
        if path.is_file():
            files.download(str(path))
